## Esto es un programa para sacar informacion para los ejercicios economicos de los hauls. Antes de empezar lo mas imortante es que te descarges la pagina de orden con todods los productos, para hacerlo haz F12 O INSPECCIONAR, vete a Elements y desde ahi en la primera linea, donde dice <html lang=``en´´> le das click derecho, copy, copy outer html.
##Creas un nuevo documento de texto .txt llamado haul.html. Haces lo mismo en la pagina de tu haul(denro de detalles de la parcela y lo llamas haulpesos.html.




In [8]:
from google.colab import files
uploaded = files.upload()

Saving haul.html.txt to haul.html (1).txt


In [9]:
from bs4 import BeautifulSoup
import pandas as pd
import re

with open("haul.html.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

order_lists = soup.select("div.order-list")
print(f"Pedidos a fábrica encontrados: {len(order_lists)}")

data = []

for order_list in order_lists:
    # Número de pedido (identifica esta compra a la fábrica)
    ordersn_tag = order_list.select_one(".order-title .ordersn")
    numero_pedido_raw = ordersn_tag.get_text(strip=True) if ordersn_tag else None
    numero_pedido = numero_pedido_raw.split(":")[-1].strip() if numero_pedido_raw else None

    order_items = order_list.select("div.order-item")

    for order_item in order_items:
        # --- Datos COMPARTIDOS por todos los productos de esta fábrica ---
        item_pay = order_item.select_one(".item-pay")
        pay_spans = item_pay.select("span") if item_pay else []

        flete_nacional_raw = pay_spans[1].get_text(strip=True) if len(pay_spans) > 1 else None
        flete_nacional = 0.0
        if flete_nacional_raw:
            match = re.search(r"([\d.]+)", flete_nacional_raw)
            flete_nacional = float(match.group(1)) if match else 0.0

        # --- Recorremos cada producto individual dentro de esta fábrica ---
        goods_list_items = order_item.select("div.goods-list")
        productos_validos = []

        for goods in goods_list_items:
            status_tag = goods.select_one(".item-status span")
            status = status_tag.get_text(strip=True) if status_tag else None

            # Si está reembolsado, lo saltamos y pasamos al siguiente producto
            if status and "refunded" in status.lower():
                continue

            codigo_art = goods.select_one(".item-detail .detail p")
            codigo = codigo_art.get_text(strip=True) if ordersn_tag else None

            imagen_art = goods.select_one(".item-detail .img-box .el-image img")
            url_imagen = imagen_art.get("src") if imagen_art else None

            nombre_tag = goods.select_one(".item-detail .detail a")
            nombre = nombre_tag.get("title", "").strip() if nombre_tag else None

            precio_tag = goods.select_one(".item-price span")
            precio_raw = precio_tag.get_text(strip=True) if precio_tag else None
            precio_match = re.search(r"([\d.]+)", precio_raw) if precio_raw else None
            precio_unitario = float(precio_match.group(1)) if precio_match else 0.0

            cantidad_tag = goods.select_one(".item-num span")
            cantidad_raw = cantidad_tag.get_text(strip=True) if cantidad_tag else None
            cantidad_match = re.search(r"(\d+)", cantidad_raw) if cantidad_raw else None
            cantidad = int(cantidad_match.group(1)) if cantidad_match else 1

            detail_link_tag = goods.select_one('.item-service a[href*="goodsDetail"]')
            goods_id = None
            if detail_link_tag and "goodsId=" in detail_link_tag.get("href", ""):
                goods_id = detail_link_tag["href"].split("goodsId=")[-1]

            productos_validos.append({
                "numero_pedido": numero_pedido,
                "codigo art": codigo,
                "imagen" : url_imagen,
                "nombre": nombre,
                "precio art": precio_unitario,
                "cantidad": cantidad,
                "status": status,
                "goods id": goods_id
            })

        # --- Repartimos el flete SOLO entre los productos válidos ---
        num_articulos = len(productos_validos)
        flete_por_articulo = (flete_nacional / num_articulos) if num_articulos > 0 else 0.0

        for producto in productos_validos:
            producto["flete N por pedido"] = flete_nacional
            producto["flete N"] = round(flete_por_articulo, 2)
            producto["P total art"] = round(
                producto["precio art"] * producto["cantidad"] + flete_por_articulo, 2
            )
            data.append(producto)

df = pd.DataFrame(data)
df

Pedidos a fábrica encontrados: 20


,numero_pedido,codigo art,imagen,nombre,precio art,cantidad,status,goods id,flete N por pedido,flete N,P total art
0,O2607161JHBT50I,OG26071628081377,https://cbu01.alicdn.com/img/ibank/O1CN01xYbE3...,外贸经典字母动物零钱男士欧尚简约两折钱包钱夹60223爆款,2.73,1,Receipted,30860152,0.49,0.49,3.22
1,O2607161JHBAKHA,OG26071687912952,https://cbu01.alicdn.com/img/ibank/O1CN01yijaC...,一件代发 新款腰花 灯芯绒 牛津布 短款钱包 男女通用钱包,2.41,1,Receipted,30860150,2.57,1.28,3.70
2,O2607161JHBAKHA,OG26071652481453,https://cbu01.alicdn.com/img/ibank/O1CN01XSJc3...,一件发 迷彩钱包 钱包 卡包迷彩短款钱包,2.41,1,Receipted,30860151,2.57,1.28,3.70
3,O26071618XU31JE,OG26071620136175,https://cbu01.alicdn.com/img/ibank/O1CN01Z0KuR...,经典十字架项链男十字架套链女ins潮欧美时髦甜酷中性卫衣链,1.61,1,Receipted,30854945,0.63,0.63,2.24
4,O26071618XUFBDJ,OG26071668208218,https://cbu01.alicdn.com/img/ibank/O1CN01WlYpD...,一件发 新款五道杠短款钱包 迷彩钱包,3.05,1,Receipted,30854944,1.29,1.29,4.34
5,O2607141J6GAYJ4,OG26071452775638,https://cbu01.alicdn.com/img/ibank/O1CN01oQt9D...,褶皱款龙珠线条适用15手机壳iPhone14软壳13防摔11promax潮款,0.41,2,Receipted,30779005,1.61,0.81,1.62
6,O2607141J6GAYJ4,OG26071496751081,https://cbu01.alicdn.com/img/ibank/O1CN01mtVcd...,路飞索隆适用16手机壳15防摔外壳iphone14promax全包12硬壳男,1.29,1,Receipted,30779006,1.61,0.81,2.10
7,O2607141J6GTN9H,OG26071482831887,https://cbu01.alicdn.com/img/ibank/O1CN01v3VJd...,跨境新款适用17promax手机壳16双层15IMD14彩银13全包12防摔x,0.48,1,Receipted,30779004,0.37,0.37,0.85
8,O2607141J6GBKKN,OG26071453019296,https://cbu01.alicdn.com/img/ibank/O1CN01JBpk4...,一件可拍美墨Baze帽dandyhats棒球帽31hat棒球帽Barbas胡须现货,8.83,1,Receipted,30779001,3.86,1.93,10.76
9,O2607141J6GBKKN,OG26071423309615,https://cbu01.alicdn.com/img/ibank/O1CN01du1R3...,一件可拍美墨Baze帽dandyhats棒球帽31hat棒球帽Barbas胡须现货,8.83,1,Receipted,30779003,3.86,1.93,10.76


## Ahora los pesos

In [3]:
from google.colab import files
uploaded_pesos = files.upload()

Saving haulpesos.html.txt to haulpesos.html.txt


In [4]:
from bs4 import BeautifulSoup
import pandas as pd
import re

lista_pesos_totales = []
lista_df_pesos = []

for nombre_archivo in uploaded_pesos.keys():
    with open(nombre_archivo, "r", encoding="utf-8") as f:
        html_peso = f.read()

    soup_peso = BeautifulSoup(html_peso, "html.parser")

    peso_total_actual = None
    for li in soup_peso.select("li"):
        if "Peso (g)" in li.get_text():
            span = li.select_one(".highlight")
            if span:
                peso_total_actual = int(span.get_text(strip=True))
            break

    print(f"{nombre_archivo} -> peso total del paquete: {peso_total_actual}g")
    lista_pesos_totales.append(peso_total_actual)

    filas = soup_peso.select("table tbody tr.ant-table-row-level-0")
    data_pesos = []
    for fila in filas:
        celdas = fila.select("td")
        codigo_articulo = celdas[1].get_text(strip=True)

        peso_span = celdas[3].select_one("span")
        peso_raw = peso_span.get_text(strip=True) if peso_span else None
        peso_match = re.search(r"(\d+)", peso_raw) if peso_raw else None
        peso_gramos = int(peso_match.group(1)) if peso_match else None

        data_pesos.append({"codigo art": codigo_articulo, "peso (g)": peso_gramos})

    lista_df_pesos.append(pd.DataFrame(data_pesos))

peso_total_paquete = sum(p for p in lista_pesos_totales if p is not None)
print(f"\nPeso total combinado de los {len(lista_pesos_totales)} paquete(s): {peso_total_paquete}g")

pesos = pd.concat(lista_df_pesos, ignore_index=True)
print(f"Artículos con peso encontrados en total: {len(pesos)}")
pesos

haulpesos.html.txt -> peso total del paquete: 5750g

Peso total combinado de los 1 paquete(s): 5750g
Artículos con peso encontrados en total: 28


,codigo art,peso (g)
0,OG26071330786609,600
1,OG26071358394335,550
2,OG26071336512754,150
3,OG26071377017026,100
4,OG26071336870749,150
5,OG26071391714975,150
6,OG26071382304941,100
7,OG26071324695080,200
8,OG26071397961256,150
9,OG26071333322068,250


##Ahora juntamos los dos df

In [10]:
df_completo = df.merge(pesos, on="codigo art", how="left")

# --- Quitamos artículos que no pertenecen a ningún paquete de este haul ---
df_completo = df_completo.sort_values(by="peso (g)", ascending=True, na_position="first")

sin_peso = df_completo["peso (g)"].isna().sum()
print(f"Artículos sin peso asignado (no estaban en los paquetes): {sin_peso}")
if sin_peso > 0:
    print(df_completo[df_completo["peso (g)"].isna()][["codigo art", "nombre"]])

df_completo = df_completo.dropna(subset=["peso (g)"]).reset_index(drop=True)

df_limpio = df_completo.drop(["nombre", "numero_pedido", "goods id", "status", "flete N por pedido"], axis=1)
df_limpio

Artículos sin peso asignado (no estaban en los paquetes): 13
          codigo art                                        nombre
28  OG26051033284789                  潘家多拉白铜镀925银色镶钻宝石爱心简约对戒时尚戒指系列
29  OG26051035045849                  潘家多拉白铜镀925银色镶钻宝石爱心简约对戒时尚戒指系列
30  OG26051000201046                  潘家多拉白铜镀925银色镶钻宝石爱心简约对戒时尚戒指系列
31  OG26051073691911                  潘家多拉白铜镀925银色镶钻宝石爱心简约对戒时尚戒指系列
32  OG26051043098071                  潘家多拉白铜镀925银色镶钻宝石爱心简约对戒时尚戒指系列
33  OG26050978362663  跨境爆款蜘蛛网iPhone17Pro手机壳16/15/14/13镭射1211XsMax/
34  OG26050997216964  跨境爆款蜘蛛网iPhone17Pro手机壳16/15/14/13镭射1211XsMax/
35  OG26050807768490             复刻 ReactX Rejuven8男女包头拖半拖休闲沙滩涉水凉鞋
36  OG26050869463369                 正确版字母渔夫帽男女网红同款时尚休闲百搭运动遮阳盆帽渔夫帽
37  OG26050810022578                    恶魔岛刺绣五角星牛仔短裤男士夏季高街宽松休闲五分裤潮
38  OG26050812205098                高端亚麻裤子男春秋季薄款新中式棉麻休闲长裤男士日系宽松直筒裤
39  OG26050835800940                高端亚麻裤子男春秋季薄款新中式棉麻休闲长裤男士日系宽松直筒裤
40  OG26050891414538                高端亚麻裤子男春秋季薄款新中式棉麻休闲长裤男士日系宽松直筒裤


,codigo art,imagen,precio art,cantidad,flete N,P total art,peso (g)
0,OG26071620136175,https://cbu01.alicdn.com/img/ibank/O1CN01Z0KuR...,1.61,1,0.63,2.24,39.0
1,OG26071687912952,https://cbu01.alicdn.com/img/ibank/O1CN01yijaC...,2.41,1,1.28,3.70,50.0
2,OG26071652481453,https://cbu01.alicdn.com/img/ibank/O1CN01XSJc3...,2.41,1,1.28,3.70,50.0
3,OG26071452775638,https://cbu01.alicdn.com/img/ibank/O1CN01oQt9D...,0.41,2,0.81,1.62,50.0
4,OG26071482831887,https://cbu01.alicdn.com/img/ibank/O1CN01v3VJd...,0.48,1,0.37,0.85,50.0
5,OG26071496751081,https://cbu01.alicdn.com/img/ibank/O1CN01mtVcd...,1.29,1,0.81,2.10,50.0
6,OG26071334443115,https://cbu01.alicdn.com/img/ibank/O1CN01UuqD9...,5.14,1,1.54,6.68,100.0
7,OG26071345532129,https://cbu01.alicdn.com/img/ibank/O1CN012G84P...,5.14,1,1.54,6.68,100.0
8,OG26071368646907,https://cbu01.alicdn.com/img/ibank/O1CN01S6yLJ...,5.14,1,1.54,6.68,100.0
9,OG26071382304941,https://cbu01.alicdn.com/img/ibank/O1CN018B3TV...,3.70,1,1.28,4.98,100.0


In [11]:
import requests
from io import BytesIO
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from PIL import Image as PILImage

excel = "articulos.xlsx"
df_limpio.to_excel(excel, index=False)
wb = load_workbook(excel)
ws = wb.active
for fila, url in enumerate(df_limpio["imagen"], start=2):   # empieza en la fila 2

    if pd.isna(url):
        continue

    try:
        # Descargar imagen
        response = requests.get(url, timeout=10)

        # Abrirla desde memoria
        img = PILImage.open(BytesIO(response.content))

        # Reducir tamaño
        img.thumbnail((80, 80))

        # Guardar temporalmente
        nombre_temp = f"temp_{fila}.png"
        img.save(nombre_temp)

        # Insertarla
        excel_img = Image(nombre_temp)
        ws.add_image(excel_img, f"C{fila}")    # <- columna donde aparecerá

        # Ajustar altura de la fila
        ws.row_dimensions[fila].height = 65

    except Exception as e:
        print(f"Error fila {fila}: {e}")
ws.column_dimensions["C"].width = 18
wb.save("haul_procesado.xlsx")
#files.download("haul_procesado.xlsx")

Error fila 2: cannot identify image file <_io.BytesIO object at 0x7fe28ca34e50>
Error fila 4: cannot identify image file <_io.BytesIO object at 0x7fe28c340270>
Error fila 6: cannot identify image file <_io.BytesIO object at 0x7fe28ca375b0>
Error fila 8: cannot identify image file <_io.BytesIO object at 0x7fe28c340270>
Error fila 10: cannot identify image file <_io.BytesIO object at 0x7fe28c340270>
Error fila 11: cannot identify image file <_io.BytesIO object at 0x7fe28d415d50>
Error fila 12: cannot identify image file <_io.BytesIO object at 0x7fe28c340270>
Error fila 13: cannot identify image file <_io.BytesIO object at 0x7fe28d415d50>
Error fila 14: cannot identify image file <_io.BytesIO object at 0x7fe28ca37380>
Error fila 17: cannot identify image file <_io.BytesIO object at 0x7fe28d415d50>
Error fila 18: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 19: cannot identify image file <_io.BytesIO object at 0x7fe28d415d50>
Error fila 20: cannot identify i

##En vez de arreglar todo a mano en excell lo automatizamos para todos los hauls.

In [12]:
eur_productos = float(input("Introduce el total pagado en EUR por los PRODUCTOS: "))
eur_envio = float(input("Introduce el total pagado en EUR por el ENVÍO: "))
envio_usd = float(input("Introduce el total pagado en USD por el ENVÍO: "))

print(f"Peso total combinado de todos los paquetes: {peso_total_paquete}g")

Introduce el total pagado en EUR por los PRODUCTOS: 214.93
Introduce el total pagado en EUR por el ENVÍO: 56.65
Introduce el total pagado en USD por el ENVÍO: 95.34
Peso total combinado de todos los paquetes: 5750g


In [13]:
df_limpio = df_limpio.rename(columns={"P total art": "P total ($)"})

# --- Precio total en euros ---
suma_usd = df_limpio["P total ($)"].sum()
df_limpio["P total (€)"] = round(df_limpio["P total ($)"] * eur_productos / suma_usd, 2)

# --- Peso real ---
suma_pesos = df_limpio["peso (g)"].sum()
df_limpio["peso real"] = round(df_limpio["peso (g)"] * peso_total_paquete / suma_pesos, 2)

# --- Flete internacional ---
df_limpio["flete Int"] = round(df_limpio["peso real"] * eur_envio / peso_total_paquete, 2)

# --- Total final ---
df_limpio["TOTAL"] = round(df_limpio["P total (€)"] + df_limpio["flete Int"], 2)

In [14]:
df_limpio["spacer"] = ""
df_limpio["venta"] = None
df_limpio["BN"] = None
df_limpio["BN%"] = None
df_limpio["TV"] = None
df_limpio["FM"] = None
df_limpio["IPM"] = None

df_limpio = df_limpio[[
    "codigo art", "imagen", "precio art", "cantidad", "flete N",
    "P total ($)", "P total (€)", "peso (g)", "peso real",
    "flete Int", "TOTAL", "spacer", "venta", "BN" , "BN%" , "TV" , "FM" , "IPM"
]]

df_limpio = df_limpio.rename(columns={"spacer": ""})

In [15]:
import requests
from io import BytesIO
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from PIL import Image as PILImage

excel = "articulos.xlsx"

# startrow=1 y startcol=1 porque Excel empieza a contar en 0:
# fila 0=1, 1=2 (D2 en fila)... columna 0=A, 1=B, 2=C, 3=D
df_limpio.to_excel(excel, index=False, startrow=1, startcol=1)

wb = load_workbook(excel)
ws = wb.active

# La columna de imagen ahora es la E (D=codigo art, E=imagen)
# Los datos empiezan en la fila 3 (fila 2 es la cabecera)
for fila, url in enumerate(df_limpio["imagen"], start=3):
    if pd.isna(url):
        continue
    try:
        response = requests.get(url, timeout=10)
        img = PILImage.open(BytesIO(response.content))
        img.thumbnail((80, 80))
        nombre_temp = f"temp_{fila}.png"
        img.save(nombre_temp)
        excel_img = Image(nombre_temp)
        ws.add_image(excel_img, f"C{fila}")
        ws.row_dimensions[fila].height = 65
    except Exception as e:
        print(f"Error fila {fila}: {e}")

ws.column_dimensions["c"].width = 18
wb.save("haul_procesado.xlsx")
#files.download("haul_procesado.xlsx")

Error fila 3: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 5: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 7: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 9: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 11: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 12: cannot identify image file <_io.BytesIO object at 0x7fe28c340310>
Error fila 13: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 14: cannot identify image file <_io.BytesIO object at 0x7fe28c340310>
Error fila 15: cannot identify image file <_io.BytesIO object at 0x7fe29fc8e070>
Error fila 18: cannot identify image file <_io.BytesIO object at 0x7fe28ca37380>
Error fila 19: cannot identify image file <_io.BytesIO object at 0x7fe28c96d9e0>
Error fila 20: cannot identify image file <_io.BytesIO object at 0x7fe28ca37380>
Error fila 21: cannot identify i

In [16]:
from openpyxl.utils import get_column_letter

columnas_resumen = ["precio art", "cantidad", "flete N", "P total ($)",
                     "P total (€)", "peso (g)", "peso real", "flete Int", "TOTAL"]

# Columna donde empieza la tabla nueva: después de la tabla principal + 2 huecos
col_label = 2 + len(df_limpio.columns) + 1   # 2 = columna B (1-indexado)

fila_cabecera = 2
fila_total = 3
fila_real = 4
fila_usd = 5
fila_eur = 6

# --- Cabecera de la tabla resumen ---
for i, nombre in enumerate(columnas_resumen, start=1):
    ws.cell(row=fila_cabecera, column=col_label + i, value=nombre)

# --- Fila "Total": suma de cada columna, de precio art hasta TOTAL ---
sumas = {col: round(df_limpio[col].sum(), 2) for col in columnas_resumen}
ws.cell(row=fila_total, column=col_label, value="Total")
for i, nombre in enumerate(columnas_resumen, start=1):
    ws.cell(row=fila_total, column=col_label + i, value=sumas[nombre])

print(f"Suma columna TOTAL: {sumas['TOTAL']} € — debería coincidir con {round(eur_productos + eur_envio, 2)} €")

# --- Fila "Real": el peso total real del paquete, bajo la columna 'peso (g)' ---
ws.cell(row=fila_real, column=col_label, value="Real")
idx_peso = columnas_resumen.index("peso (g)") + 1
ws.cell(row=fila_real, column=col_label + idx_peso, value=peso_total_paquete)

# Posiciones de las 3 últimas columnas, reutilizadas como "productos / envío / total"
idx_c1 = columnas_resumen.index("peso real") + 1
idx_c2 = columnas_resumen.index("flete Int") + 1
idx_c3 = columnas_resumen.index("TOTAL") + 1

# --- Fila "USD": productos, envío y total en dólares ---
ws.cell(row=fila_usd, column=col_label, value="USD")
productos_usd = round(df_limpio["P total ($)"].sum(), 2)
total_usd = round(productos_usd + envio_usd, 2)
ws.cell(row=fila_usd, column=col_label + idx_c1, value=productos_usd)
ws.cell(row=fila_usd, column=col_label + idx_c2, value=envio_usd)
ws.cell(row=fila_usd, column=col_label + idx_c3, value=total_usd)

# --- Fila "EUR real": los inputs originales en euros ---
ws.cell(row=fila_eur, column=col_label, value="EUR real")
eur_total = round(eur_productos + eur_envio, 2)
ws.cell(row=fila_eur, column=col_label + idx_c1, value=eur_productos)
ws.cell(row=fila_eur, column=col_label + idx_c2, value=eur_envio)
ws.cell(row=fila_eur, column=col_label + idx_c3, value=eur_total)

wb.save("haul_procesado.xlsx")
#files.download("haul_procesado.xlsx")

Suma columna TOTAL: 271.51 € — debería coincidir con 271.58 €


In [17]:
from openpyxl.styles import Border, Side

grueso = Side(style="thick")

def borde_exterior(ws, fila_inicio, fila_fin, col_inicio, col_fin):
    for fila in range(fila_inicio, fila_fin + 1):
        for col in range(col_inicio, col_fin + 1):
            celda = ws.cell(row=fila, column=col)
            bordes = {}
            if fila == fila_inicio:
                bordes["top"] = grueso
            if fila == fila_fin:
                bordes["bottom"] = grueso
            if col == col_inicio:
                bordes["left"] = grueso
            if col == col_fin:
                bordes["right"] = grueso
            if bordes:
                celda.border = Border(**bordes)

# Tabla 1: desde D2 hasta la última fila/columna de datos
fila_fin_t1 = 2 + len(df_limpio)          # fila 2 (cabecera) + nº de filas de datos
col_fin_t1 = 1 + len(df_limpio.columns)   # D=4 (1-indexed) + nº columnas - 1
borde_exterior(ws, 2, fila_fin_t1, 2, col_fin_t1)

# Tabla 2: desde su cabecera hasta la fila "EUR real"
borde_exterior(ws, fila_cabecera, fila_eur, col_label, col_label + len(columnas_resumen))

wb.save("haul_procesado.xlsx")

In [18]:
from openpyxl.styles import PatternFill, Font
from openpyxl import Workbook
from openpyxl.styles import Alignment

relleno_cabecera = PatternFill(start_color="9A0067", end_color="B40078", fill_type="solid")
fuente_cabecera = Font(color="FFFFFF", bold=True)

relleno_lado = PatternFill(start_color="E60073", end_color="E60073", fill_type="solid")
fuente_lado = Font(color="000000", bold=False)

# Lado tabla 1
for row in range(fila_cabecera + 1, fila_fin_t1 + 1):
    celda = ws.cell(row=row, column=2)
    celda.fill = relleno_lado
    celda.font = fuente_lado
# Cabecera tabla 1 (fila 2, desde columna B)
for col in range(2, 2 + len(df_limpio.columns)):
    celda = ws.cell(row=2, column=col)
    celda.fill = relleno_cabecera
    celda.font = fuente_cabecera

# Lado tabla 2
for row in range(fila_cabecera + 1, fila_cabecera + 5):
    celda = ws.cell(row=row, column=col_label)
    celda.fill = relleno_lado
    celda.font = fuente_lado
# Cabecera tabla 2
for col in range(col_label + 1, col_label + 1 + len(columnas_resumen)):
    celda = ws.cell(row=fila_cabecera, column=col)
    celda.fill = relleno_cabecera
    celda.font = fuente_cabecera


# Centrar el texto horizontal y verticalmente
for row in range(1, fila_fin_t1 + 1):
  for col in range(1, 13 + len(df_limpio.columns)):
    celda = ws.cell(row=row, column=col)
    celda.alignment = Alignment(horizontal="center", vertical="center")

# Recorrer todas las filas de la columna 1
for row in range(1, fila_fin_t1 + 1):
    # 1. Seleccionar la celda de la columna 1 en la fila actual
    celda_col1 = ws.cell(row=row, column=1)

    # 2. Asignar el texto con un espacio en blanco
    celda_col1.value = " "

wb.save("haul_procesado.xlsx")

In [19]:
ancho_col = (81 - 5) / 7        # ≈ 8.43
alto_fila = 81 * 0.75          # ≈ 48.4

ultima_col = col_label + len(columnas_resumen)  # la columna más a la derecha usada
ultima_fila = max(fila_fin_t1, fila_eur)         # la fila más abajo usada

for col in range(1, ultima_col + 1):
    letra = get_column_letter(col)
    ws.column_dimensions[letra].width = ancho_col

for fila in range(1, ultima_fila + 1):
    ws.row_dimensions[fila].height = alto_fila

wb.save("haul_procesado.xlsx")

In [20]:
files.download("haul_procesado.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>